In [1]:
import pandas as pd

In [21]:
results1 = pd.read_csv("../data/results/post_level_results.csv")
results2 = pd.read_csv("../data/sentiment_results/post_level_results_2.csv")
results3 = pd.read_csv("../data/sentiment_results/post_level_results_3.csv")

results1 = results1.rename(columns={"llm_sentiment": "llm_label_first", "llm_confidence": "llm_conf_first", "llm_reasoning": "llm_reason_first"})
results2 = results2.rename(columns={ "llm_sentiment": "llm_label_second", "llm_confidence": "llm_conf_second", "llm_reasoning": "llm_reason_second" })
results3 = results3.rename(columns={ "llm_sentiment": "llm_label_third", "llm_confidence": "llm_conf_third", "llm_reasoning": "llm_reason_third"})

results = results1.merge(results2[["id", "llm_label_second", "llm_conf_second", "llm_reason_second"]], on="id", how="left")
results = results.merge( results3[["id", "llm_label_third", "llm_conf_third", "llm_reason_third"]], on="id", how="left")

results = results.reset_index(drop=True)

In [22]:
results.head(1)

,id,subreddit_name,source_set,score,created_utc,clean_post_text,overall_label,overall_negative,overall_neutral,overall_positive,...,topic_strongly_positive,llm_label_first,llm_conf_first,llm_reason_first,llm_label_second,llm_conf_second,llm_reason_second,llm_label_third,llm_conf_third,llm_reason_third
0,1rt6kxz,marketing,low,15,1773452773,"How do you manage conference event ROI? Our company goes to 10+ conferences per year plus dozens of virtual and sponsored events. I’m tasked to create a dashboard that over sees the outcome (MQLs Meetings, Emerging opportunities and closed deals) of each event that stays up to days as leads move through the pipeline. We use hubspot for our CRM where we have list segments imported from each event. How do marketers track event ROI assuming that a customer life cycle is 7+ months? I’ve tried using Google sheets but it gets complex and unstable.",neutral,0.035945,0.871742,0.092313,...,0.000275,neutral,0.8,The post mentions using HubSpot for CRM and list segments from events but does not express a clear positive or negative opinion about HubSpot.,neutral,0.9,The post describes using HubSpot for CRM and event tracking without expressing clear positive or negative opinions about the platform.,neutral,0.95,The post is mainly asking for advice on managing event ROI using HubSpot and does not express satisfaction or frustration with HubSpot itself.


In [23]:
# Teeme topic based meelestatuse viis klassi kolmeks klassiks

# Võtame "strongly negative" ja "negative" kokku -> "negative"
# Muudame "negative or neutral" -> "neutral"
# Võtame "strongly positive" ja "positive" kokku -> "positive"
def topic_to_3(label):
    if label in ["strongly negative", "negative"]:
        return "negative"
    if label == "negative or neutral":
        return "neutral"
    if label in ["positive", "strongly positive"]:
        return "positive"
    return label

results["topic_label_3class"] = results["topic_label"].apply(topic_to_3)

## Mudelite klassifitseerimise jaotused

# Mudelite klassifitseerimise jaotused

In [24]:
import pandas as pd

labels = ["negative", "neutral", "positive"]

def precent_distribution(model_labels):
    counts = model_labels.value_counts(normalize=True).reindex(labels, fill_value=0)
    return (counts * 100).round(1)

table = pd.DataFrame({
    "Üldine mudel": precent_distribution(results["overall_label"]),
    "Teemapõhine mudel": precent_distribution(results["topic_label_3class"]),
    "LLM (GPT-4.1-mini)": precent_distribution(results["llm_label_first"]),
    "LLM kontekstiga": precent_distribution(results["llm_label_second"]),
    "LLM rohkem kontekstiga": precent_distribution(results["llm_label_third"]),
})


table.index = ["Negatiivne", "Neutraalne", "Positiivne"]
table = table.astype(str) + "%"
table

,Üldine mudel,Teemapõhine mudel,LLM (GPT-4.1-mini),LLM kontekstiga,LLM rohkem kontekstiga
Negatiivne,21.0%,11.8%,27.6%,18.7%,26.9%
Neutraalne,66.0%,69.4%,46.0%,57.8%,62.4%
Positiivne,13.0%,18.7%,26.4%,23.5%,10.7%


In [34]:
import pandas as pd

table = pd.DataFrame({
    "Mudelid": [
        "Üldine vs. teemapõhine",
        "LLM v1 vs. v2",
        "LLM v1 vs. v3",
        "LLM v2 vs. v3"
    ],
    "Erinevuste arv": [329, 218, 281, 302],
    "Erinevuste osakaal (%)": ["26,0%", "17,2%", "22,2%", "23,9%"]
})

display(table.style.hide(axis="index"))

Mudelid,Erinevuste arv,Erinevuste osakaal (%)
Üldine vs. teemapõhine,329,"26,0%"
LLM v1 vs. v2,218,"17,2%"
LLM v1 vs. v3,281,"22,2%"
LLM v2 vs. v3,302,"23,9%"


In [25]:
crosstab_pairs = [("overall_label", "topic_label_3class"), ("llm_label_first","llm_label_second"), ("llm_label_first","llm_label_third"), ("llm_label_second","llm_label_third")]
for i, j in crosstab_pairs:
    print(f"\nDifferences: {i} vs {j}")
    display(pd.crosstab(results[i], results[j]))


Differences: overall_label vs topic_label_3class


topic_label_3class,negative,neutral,positive
overall_label,,,
negative,119,145,2
neutral,30,694,111
positive,1,40,124



Differences: llm_label_first vs llm_label_second


llm_label_second,negative,neutral,positive
llm_label_first,,,
negative,224,126,0
neutral,13,548,21
positive,0,58,276



Differences: llm_label_first vs llm_label_third


llm_label_third,negative,neutral,positive
llm_label_first,,,
negative,307,43,0
neutral,31,547,4
positive,3,200,131



Differences: llm_label_second vs llm_label_third


llm_label_third,negative,neutral,positive
llm_label_second,,,
negative,220,17,0
neutral,119,611,2
positive,2,162,133


## Kuldstandardiga täpsuse mõõtmine

In [26]:
gold = pd.read_excel("../data/gold_standard_filled.xlsx")
gold = gold.merge(results[["id", "source_set", "overall_label", "topic_label_3class", "llm_label_first", "llm_label_second", "llm_label_third"]], on="id")


In [27]:
from sklearn.metrics import accuracy_score

high = gold[gold["source_set"] == "high"]
low = gold[gold["source_set"] == "low"]

overall_accuracy = accuracy_score(gold["gold_label"], gold["overall_label"])
topic_accuracy = accuracy_score(gold["gold_label"], gold["topic_label_3class"])
llm_accuracy_first = accuracy_score(gold["gold_label"], gold["llm_label_first"])
llm_accuracy_second = accuracy_score(gold["gold_label"], gold["llm_label_second"])
llm_accuracy_third = accuracy_score(gold["gold_label"], gold["llm_label_third"])

overall_high = accuracy_score(high["gold_label"], high["overall_label"])
topic_high = accuracy_score(high["gold_label"], high["topic_label_3class"])
llm_high_1 = accuracy_score(high["gold_label"], high["llm_label_first"])
llm_high_2 = accuracy_score(high["gold_label"], high["llm_label_second"])
llm_high_3 = accuracy_score(high["gold_label"], high["llm_label_third"])



overall_low = accuracy_score(low["gold_label"], low["overall_label"])
topic_low = accuracy_score(low["gold_label"], low["topic_label_3class"])
llm_low_1 = accuracy_score(low["gold_label"], low["llm_label_first"])
llm_low_2 = accuracy_score(low["gold_label"], low["llm_label_second"])
llm_low_3 = accuracy_score(low["gold_label"], low["llm_label_third"])


table = pd.DataFrame({
    "Mudel": ["Üldine", "Teemapõhine", "LLM", "LLM(kontekst)", "LLM(rohkem konteksti)"],
    "Kokku": [ round(overall_accuracy * 100, 1), round(topic_accuracy * 100, 1), round(llm_accuracy_first * 100, 1) , round(llm_accuracy_second * 100, 1), round(llm_accuracy_third * 100, 1)],
    "Kõrge": [round(overall_high * 100, 1),round(topic_high * 100, 1), round(llm_high_1 * 100, 1), round(llm_high_2 * 100, 1), round(llm_high_3 * 100, 1)],
    "Madal": [round(overall_low * 100, 1), round(topic_low * 100, 1),round(llm_low_1 * 100, 1), round(llm_low_2 * 100, 1), round(llm_low_3 * 100, 1)],
})

table["Kokku"] = table["Kokku"].astype(str) + "%"
table["Kõrge"] = table["Kõrge"].astype(str) + "%"
table["Madal"] = table["Madal"].astype(str) + "%"

table

,Mudel,Kokku,Kõrge,Madal
0,Üldine,60.0%,60.4%,58.5%
1,Teemapõhine,56.3%,56.5%,55.4%
2,LLM,70.2%,71.3%,66.2%
3,LLM(kontekst),70.2%,68.7%,75.4%
4,LLM(rohkem konteksti),83.7%,83.9%,83.1%


In [28]:
pd.set_option('display.max_colwidth', None)

In [29]:
gold[(gold["gold_label"] == "neutral") & (gold["llm_label_third"] != "neutral")]

id  \
10   1olfl7i   
65   1rx5qss   
90   1p37xy4   
131  1rb5rdf   
135  1qamtk0   
152  1qngsvh   
162  1ryehxp   
170  1q98bxz   
176  1qpgnyy   
186  1sc1h1d   
188  1r1hwf6   
189  1onsz8r   
197  1p5zn0q   
198  1r7aafq   
205  1s8bv0v   
217  1sbye9v   
229  1q6gkwj   
240  1rs1uqy   
248  1qpfbsf   
249  1rrqjvm   
253  1oyfsps   
272  1p9kzn9   
275  1ol56o9   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

## Erinevuste näited

In [30]:
overall_topic_differneces = results[results["overall_label"] != results["topic_label_3class"]].copy()
print("Erinevusi kokku:", overall_topic_differneces.shape[0])

label_order = ["negative", "neutral", "positive"]

samples = []

for overall in label_order:
    for topic in label_order:
        # skip kui sama label (ei ole "difference")
        if overall == topic:
            continue
        
        subset = results[
            (results["overall_label"] == overall) &
            (results["topic_label_3class"] == topic)
        ]
        
        if len(subset) > 0:
            sample = subset.sample(1, random_state=42)
            samples.append(sample)

# liidame kõik kokku
pair_samples_1 = pd.concat(samples)

display(pair_samples_1[[
    "clean_post_text",
    "overall_label",
    "topic_label_3class"
]])

Erinevusi kokku: 329


,clean_post_text,overall_label,topic_label_3class
484,"whats hubspot bad at Hi, thinking about using hubspot, is there anything its not great at? Can automate some things outside of hubspot if its weak in some areas. Thanks!",negative,neutral
1117,"It’s time to actually use HubSpot’s lead capture tools I can’t tell you how many HubSpot portals I’ve opened where the lead capture tools are just sitting there. Forms not connected. Pop-ups disabled. Tracking code missing. And then people wonder why nothing new is coming into their CRM. I mean, you don’t need a huge strategy to start generating leads, you just need to use the basics that are already built in. Here’s where I’d start: * Install the tracking code on every page! * Add a simple form to your highest-traffic page (even a “Request pricing” or “Contact us” form) * Test out the meeting widget * Try one pop-up or slide-in with a clear helpful offer * Track what’s actually converting, not just what’s getting clicks. These are very easy tools to set up and use but they can make a huge difference. You’d be surprised how much growth comes from simply turning on what you already have. As someone who used to work at HubSpot and now helps clients set up and optimise their portals at [Baskey Digital]( I still find that the simplest setups often perform best. Any other HubSpot features you’ve seen make the biggest impact when a team finally started using it?",negative,positive
1106,"HubSpot API 404 on new Bach List Membership endpoint Hi all, Recently HubSpot announced a new batch endpoint for checking association list memberships by POST'ing to /crm/v3/lists/records/memberships/batch/read However, for some reason, I keep 404'ing to that endpoint. Does anyone else have this issue? Did anyone find a fix? Find the announcement [here]( 3rd heading. Find the API docs relevant to the new batch endpoint [here]( I'm experienced in building API-heavy workflows, and I am very confident my setup is correct. See attached screenshots for reference. I contacted Support, they replied quickly and said: ""Based on the error details, in fact it just looks like the endpoint is not available yet in your workspace. This usually happens when the endpoint is newly released but not fully rolled out to all portals."" In my experience, if they do a staged rollout they'd mention it clearly in communications and the error would not be a 404 but a 401 or 403. To me it seems like this endpoint should be available. Anyone experiencing a similar issue?",neutral,negative
891,Upgraded to Enterprise (Marketing and Content Hub) - what should be my first move? Upgraded from Pro to Enterprise (or whatever the level below enterprise is called). What are the obvious things I now have access to that I should take advantage of?,neutral,positive
975,"Database Damage on a Regular Basis I love AI, it's a tool that has improved my life in many ways when it comes to efficiency and workflow. I have never had so many disappointments though since I started with HubSpot around July or August. I am so tired of the unacceptable untrained AI changing phone numbers and deleting contacts that I actually work with. Rant over. I'm ready to ditch it.",positive,negative
659,"A question about HubSpot campaigns and UTM tracking Hi everyone, I hope you're having wonderful weeks. We're just starting to use campaigns in our HubSpot and I had a question. Say we have a campaign called ""ntc"" and I set the utm\_campaign parameter in the campaign manager to ""ntc"", do I then need to create the URLs inside the campaign manager, or can I carry on using a UTM spreadsheet, so long as I include the utm\_campaign as ""ntc"". Ideally, we'd like all of our PPC, email and outreach to have this UTM in for tracking.",positive,neutral


In [31]:
llm_1_2_differences = results[results["llm_label_first"] != results["llm_label_second"]].copy()
print("Erinevusi kokku:", llm_1_2_differences.shape[0])


samples = []

for overall in label_order:
    for llm in label_order:
        if overall == llm:
            continue
        
        subset = results[
            (results["llm_label_first"] == overall) &
            (results["llm_label_second"] == llm)
        ]
        
        if len(subset) > 0:
            samples.append(subset.sample(1, random_state=42))

pair_samples_2 = pd.concat(samples)

display(pair_samples_2[[
    "clean_post_text",
    "llm_label_first",
    "llm_label_second"
]])

Erinevusi kokku: 218


,clean_post_text,llm_label_first,llm_label_second
794,"Question about HubSpot email analytics after users click a button link Hey everyone, We’re using HubSpot as our CRM and recently ran an email project where recipients could choose one of three organizations to support by clicking a button in the email. Each button linked to a specific page on our website. Everything looked great at first, high open rates and strong interaction in the email. But once users landed on the website, the analytics stopped registering properly. In many cases, we couldn’t see which organization each person selected. We tested the setup beforehand, including private/incognito windows and cookie rejection, and it seemed to work fine in all test cases. However, during the live send, HubSpot analytics only worked for about half of the users, the other half wasn’t tracked as expected. Has anyone else experienced something like this or found a reliable way to capture which link users chose after clicking from the email? Thanks for your Feedback!",negative,neutral
958,"Where did the last Salesforce sync date go? In one of the portals that I’m working in, we stopped seeing this property. Anyone else experiencing that?",neutral,negative
165,"Is PPC the least AI-affected niche in digital marketing? I'm in my last year of a Bachelor of Arts degree majoring in Media and Communications and am looking for advice on my next steps. I'm thinking of focusing on digital marketing, specifically PPC, which, from what I've gathered, is one of the niches in digital marketing least affected by AI. Mainly because unlike SEO which is being killed by Google AI summaries, paid ads espeically google ads are less affected. Also in the future demand for PPC services may increase as AI chatbots like ChatGPT start to incorporate paid ads into their platforms. Am I on the right track in my assessment of the industry that PPC is the way to go? I'm thinking that after I graduate, I will do a few online courses (Google Skillshop, Hubspot, Meta) before applying for one of the big digital marketing agencies in my city (Melbourne, Australia). Apparently the pay is crap and it's long hours so hopefully I can get a in-house gig after a few years. Any advice would be appreciated, cheers.",neutral,positive
6,"How to know if you’re ready to transition to Marketing Operations from Email Marketing? I have 5 years of full time marketing experience, 4 of those being email-specific roles in lifecycle and demand gen. Each role I gain more and more ownership and the email programs I manage have grown in complexity. Currently, I’m at a B2B SaaS company and use HubSpot - building workflows, segments/lists, and email builds mainly; then we use Looker for reporting. I’m able to do a lot that others usually just hand off to MOps such as investigating problems/inconsistencies in lists or syncing Amplitude cohorts I build for audience lists for my evergreen automations. If it’s something particularly complicated and I can’t solve it, then I’ll hand off to MOps. As someone who has gotten burnt out by the strategy/brainstorming side of email marketing, these experiences are what make me wonder I may be a good fit to transition to MOps. I know my MOps team sees me as an employee who can generally handle their own compared to other employees more dependent on them, but I’m not sure if that necessarily means I am ready to be in MOPs full time. I sometimes look at MOps as all-knowing figures. Obviously they’re not but I’ll go to them when I really can’t figure something out, they solve it, and I am in awe that they were able to fix/troubleshoot as needed. It makes me wonder “well, if I still have to depend on MOps, even if it’s less than others in similar roles as me, am I actually ready to consider applying for such roles..?” Not sure if it’s impostor syndrome, if MOps Specialists/Managers do genuinely know a lot more than I currently do before starting it full time, or if there’s a TON of training happening b

In [32]:
llm_1_2_differences = results[results["llm_label_first"] != results["llm_label_third"]].copy()
print("Erinevusi kokku:", llm_1_2_differences.shape[0])


samples = []

for overall in label_order:
    for llm in label_order:
        if overall == llm:
            continue
        
        subset = results[
            (results["llm_label_first"] == overall) &
            (results["llm_label_third"] == llm)
        ]
        
        if len(subset) > 0:
            samples.append(subset.sample(1, random_state=42))

pair_samples_2 = pd.concat(samples)

display(pair_samples_2[[
    "clean_post_text",
    "llm_label_first",
    "llm_label_third"
]])

Erinevusi kokku: 281


,clean_post_text,llm_label_first,llm_label_third
1095,"Sudden increase in spam form submissions I have seen across several portals over the last few days multiple form submissions come in from spam sources, but it is really weird. They company, first, and last name will be a jumble of upper and lower case letters, but with what looks like a real email and phone number. Some of the emails in are legitimate. while others aren't. Has anyone else gotten this in their portals?",negative,neutral
1094,"Compliance Nightmares for Event Management Hi everyone. I could use some brain power to solve this one. **For context:** I'm a part of a highly regulated industry that requires every tool I use to be approved, along with all the features within the tool. Every tool that communicates with the public must meet this criteria: it can be monitored by the broker, it can be retrieved by the broker, it is archived on a secure server that the broker has access to. Because the broker is massive and manages a lot of other businesses much smaller than ours, we're usually ahead of the curve on discovering innovative technology, and our products are almost never on their approval roadmap. **The current problem**: Our business has been conducting research on event hosting to see how profitable these could be for prospecting. At first, we were using Eventbrite, integrated with Hubspot and everything was easy and effective. However, after about 2-3 events, we learned that Eventbrite does not meet the policy requirements mentioned before and were forced to disable our account. This is because there are some communications with Eventbrite that are sent from and recorded on their servers, that cannot be disabled. Because this isn't on their product road map, my solution suggestions to this problem were not considered. **Our solution attempts thus far:** \- The broker recommended one software, [SimpleEvent.io]( (since this uses all native Hubspot tools). However, when trying this tool out, it was not effective. Creating an event was not possible, even after talking to ""support"" (there is one person managing all communication for this tool). \- I've attempted a few workarounds within Hubspot itself. * First I attempted to create fields on the contact level that track event details: last event registered (single line text), last event attendance status (dropdown select), last event registration date (date), and the same fields for ""original event."" I then created forms for registration, cancelation, and check in as well as workflows that update these fields and prompt actions such as confirmation emails, reminders, and follow up emails. These workflows function off triggers for the ""last registration name"" and ""last registration status"". Lastly, I've created segments for ""invited"", ""registered"", ""attended"", and ""leads"" based on forms completed, emails received, and check ins completed. This worked pretty well for the first 2 events, however now I'm facing a new problem. **The next problem**: In our future events, these fields will certainly cause issues. We plan to have 1-2 events per month and some of the target audiences will overlap. So this means **when someone registers for more than one event** it could cause reporting issues, segment problems, workflow/communication problems, etc*.* For example: Client A registers today for an event on Nov 15. The workflow triggers a confirmation email, they're added to a segment for reminder emails and if they are checked in at the event, they'll receive a follow up email. The same day the same client registers for an event on Dec 4. The ""last event registered"" field is updated to the new event. Now when they are checked in, Hubspot doesn't know how to read that the ""last event"" was 2 separate instances or which event we're referring to. There are other rippling problems that stem from this. **My other solution attempts**: Unfortunately, the current event object is not robust enough to manage this

In [33]:
llm_1_2_differences = results[results["llm_label_second"] != results["llm_label_third"]].copy()
print("Erinevusi kokku:", llm_1_2_differences.shape[0])


samples = []

for overall in label_order:
    for llm in label_order:
        if overall == llm:
            continue
        
        subset = results[
            (results["llm_label_second"] == overall) &
            (results["llm_label_third"] == llm)
        ]
        
        if len(subset) > 0:
            samples.append(subset.sample(1, random_state=42))

pair_samples_2 = pd.concat(samples)

display(pair_samples_2[[
    "clean_post_text",
    "llm_label_second",
    "llm_label_third"
]])

Erinevusi kokku: 302


,clean_post_text,llm_label_second,llm_label_third
2,"Marketing ,Sales alignment is killing me. Anyone actually solved this? Every quarter it's the same fight. Marketing says we delivered X leads. Sales says leads are garbage. We're using HubSpot for marketing automation and Salesforce for CRM but there's this black hole between 'MQL' and 'closed won' where nobody knows what happened. Attribution is a mess. Sales blames marketing, marketing blames sales. How have you actually solved this beyond just 'communicate better'?",negative,neutral
649,"Built a small app to convert CSV contact list exported from Hubspot to LinkedIn audience format I recently decided to test LinkedIn Ads. Pretty quickly, I realized you can’t take a CSV exported from HubSpot and import it directly into LinkedIn. The formats just don’t match, and LinkedIn doesn’t even provide an interface to map columns, which felt very weird to me. I also didn’t want to use the native LinkedIn integration, since the contact list was far larger than my marketing contact tier in HubSpot. I searched internet for a solution but couldn’t find anything that solved the problem cleanly. So I built a small app to handle it and I’m sharing it here in case someone runs into the same issue: [uibakery.io/templates/hubspot-to-linkedin-csv-converter]( Everything runs entirely in your browser. No files are uploaded and no data is sent to any server, so your customer data stays safe. Hope it would help anyone.",neutral,negative
1001,"Phone for HubSpot that actually syncs properly? **Edit:** Quick update in case anyone else is in the same boat. We ended up trying out [**Aircall**]( after a few folks recommended it, and it’s been a massive improvement over what we were doing before. The HubSpot sync is actually reliable (finally), and calls are logging on their own with notes + recordings without me having to double-enter everything. We've been using HubSpot for about 8 months now and overall it's been great for managing our pipeline. However, our current phone setup is killing productivity. Currently using a system where calls don't log automatically so I'm manually entering everything after each call. Looking for a phone for HubSpot that integrates well: click-to-dial from contacts, automatic call logging with recordings, and customer info popping up during calls. We're 6 people now, growing to 10 soon. Right now we just use cell phones and type everything into HubSpot manually. Tried the built-in calling but quality was rough and kept dropping. Few questions for anyone who's solved this: * Does your system show HubSpot contact info during calls or do you switch tabs? * How's call quality for remote workers? (half our team is remote) * Can it auto-create tickets from calls? * Any integration issues or does it stay stable? Any leads would be appreciated.",neutral,positive
1006,"Built a Chrome extension to customize HubSpot’s UI colors I recently found out that HubSpot doesn’t let you change any of the internal UI colors… which felt kind of crazy. So I built a small personal project to fix that. **HubSpot UI Color Changer** Lets you customize your sidebar, top bar, icon colors, hover states, active colors, etc. It’s lightweight, simple, and just makes HubSpot feel a bit more “yours.” Chrome Web Store: [ If you try it out, I’d love feedback",positive,negative
1248,Google Business Profile Would anyone be interested in being able to manage their google business profile from within hubspot?,positive,neutral
